In [0]:
import sys
sys.path.insert(0, "/Workspace/Users/jean.zelada06@gmail.com/.bundle/bundle_dev_finpay/dev/files/src")
 
from ingestion_functions import (
    load_archetypes,
    add_audit_columns,
    write_to_delta_batch,
)
 
from pyspark.sql.functions import col, split, concat_ws, lit, current_timestamp


In [0]:
METADATA_PATH = (
    "/Volumes/fintech_finpay/default/vol_config/ingestion_archetypes.json"
)
 
all_active = load_archetypes(spark, METADATA_PATH)

# Extraer solo el arquetipo de users
archetype = next(
    (a for a in all_active if a["source_name"] == "users"),
    None
)
 
if archetype is None:
    raise ValueError(
        "Arquetipo 'users' no encontrado o no está activo en el JSON. "
        "Verifica que active=true en ingestion_archetypes.json."
    )
 
print(f"Arquetipo cargado: {archetype['source_name']}")
print(f"bad_records_path : {archetype['bad_records_path']}")
print(f"target_table     : {archetype['target_catalog']}.{archetype['target_schema']}.{archetype['target_table']}")


2026-05-27 04:06:16,204 [INFO] Leyendo metadata desde: /Volumes/fintech_finpay/default/vol_config/ingestion_archetypes.json
2026-05-27 04:06:16,941 [INFO] Arquetipos — total: 3 | activos: 3 | inactivos: 0


Arquetipo cargado: users
bad_records_path : /Volumes/fintech_finpay/default/vol_metadata/bad_records/users/
target_table     : fintech_finpay.bronze.users


In [0]:
BAD_RECORDS_PATH = "/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/"
 
df_bad = (
    spark.read
    .format("json")
    .option("multiLine", "false")
    .load(BAD_RECORDS_PATH)
)
 
total_bad = df_bad.count()
print(f"Bad records leídos : {total_bad}")
print(f"Columnas disponibles: {df_bad.columns}")
 
# Vista previa
display(df_bad.limit(5))


Bad records leídos : 333
Columnas disponibles: ['path', 'reason', 'record']


path,reason,record
/Volumes/fintech_finpay/default/vol_landing/users/users_20240101.txt,org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper,USR-004053|Jaqueline|Escobar Sanches|91861579|jaqueline.escobar.sa73@gmail.com|(51)959262812|AR|PREMIUM|2020-04-04
/Volumes/fintech_finpay/default/vol_landing/users/users_20240101.txt,org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper,USR-003438|Cristobal|Pascual Mondragón Villaseñor|87398842|cristobal.pascual.mo23@outlook.com|956555897|AR|premium|2023-01-28
/Volumes/fintech_finpay/default/vol_landing/users/users_20240101.txt,org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper,USR-002758|Ana|María Sáenz|23011033|anamaría.sáenz31@gmail.com|+51913647507|MX|estandar|2020-11-20
/Volumes/fintech_finpay/default/vol_landing/users/users_20240101.txt,org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper,USR-002472|Rosalia|Alonso Concepción|2999933|rosaliaalonso.concep45@hotmail.com|+51948524865|MX|premium|2021-04-05
/Volumes/fintech_finpay/default/vol_landing/users/users_20240101.txt,org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper,USR-003586|Miriam|Elizondo|66749009|miriamelizondo42@yahoo.com|(51)936451132|PE|nuevo|2021-01-19


In [0]:
DELIMITER = archetype.get("delimiter", "|")
 
df_split = (
    df_bad
    .withColumn(
        "parts",
        split(col("record"), f"\\{DELIMITER}")
    )
)


2026-05-27 04:06:19,179 [INFO] Received command c on object id p0


In [0]:
df_fixed = (
    df_split
    .select(
        col("parts")[0].alias("user_id"),
 
        concat_ws(
            " ",
            col("parts")[1],
            col("parts")[2]
        ).alias("full_name"),
 
        col("parts")[3].alias("document_id"),
        col("parts")[4].alias("email"),
        col("parts")[5].alias("phone"),
        col("parts")[6].alias("country"),
        col("parts")[7].alias("segment"),
        col("parts")[8].alias("registration_date")
    )
)
 
print(f"Registros reconstruidos: {df_fixed.count()}")
display(df_fixed.limit(10))


Registros reconstruidos: 333


user_id,full_name,document_id,email,phone,country,segment,registration_date
USR-004053,Jaqueline Escobar Sanches,91861579,jaqueline.escobar.sa73@gmail.com,(51)959262812,AR,PREMIUM,2020-04-04
USR-003438,Cristobal Pascual Mondragón Villaseñor,87398842,cristobal.pascual.mo23@outlook.com,956555897,AR,premium,2023-01-28
USR-002758,Ana María Sáenz,23011033,anamaría.sáenz31@gmail.com,+51913647507,MX,estandar,2020-11-20
USR-002472,Rosalia Alonso Concepción,2999933,rosaliaalonso.concep45@hotmail.com,+51948524865,MX,premium,2021-04-05
USR-003586,Miriam Elizondo,66749009,miriamelizondo42@yahoo.com,(51)936451132,PE,nuevo,2021-01-19
USR-001828,Dra. Yuridia Munguía,18876507,dra.yuridia.munguía24@hotmail.com,+51965569440,AR,estandar,2021-03-13
USR-001210,Dra. Soledad Garica,0244464,dra..soledad.garica18@outlook.com,,PE,estandar,2021-06-03
USR-001632,Salma Matías,59909962,salma.matías48@hotmail.com,,MX,estandar,24/03/2021
USR-003043,Rosa Mares,81501158,rosamares89@hotmail.com,968426834,AR,nuevo,05/09/2022
USR-000713,Abelardo Mares,06068699,abelardomares38@yahoo.com,+51920662614,CL,premium,2021-04-28


In [0]:
nulls_pk = df_fixed.filter(col("user_id").isNull()).count()
 
if nulls_pk > 0:
    print(f"⚠️  {nulls_pk} registros con user_id nulo — revisando...")
    display(df_fixed.filter(col("user_id").isNull()))
    # Filtrar filas con PK nula (probablemente el header del archivo)
    df_fixed = df_fixed.filter(col("user_id").isNotNull())
    print(f"   Registros tras filtrar nulos en PK: {df_fixed.count()}")
else:
    print(f"Validación PK: sin nulos en user_id")


Validación PK: sin nulos en user_id


In [0]:
df_audited = (
    add_audit_columns(df_fixed, archetype)
    .withColumn("_source_file", lit(BAD_RECORDS_PATH))  # origen = carpeta bad records
)
 
print("Columnas de auditoría añadidas")
print(f"Columnas finales: {df_audited.columns}")


2026-05-27 04:06:20,924 [INFO]   [OBS] Columnas de observabilidad añadidas ✓


Columnas de auditoría añadidas
Columnas finales: ['user_id', 'full_name', 'document_id', 'email', 'phone', 'country', 'segment', 'registration_date', '_ingestion_ts', '_source_name', '_source_format', '_schema_version', '_source_file']


In [0]:

#LLAMADA A LA FUNCIÓN DE INSERSION DE REGISTROS
records_written = write_to_delta_batch(spark, df_audited, archetype)
 
print()
print("=" * 60)
print("    REPORTE REPROCESO BAD RECORDS — users")
print("=" * 60)
print(f"  Bad records leídos   : {total_bad}")
print(f"  Registros insertados : {records_written}")
print(f"  Tabla destino        : {archetype['target_catalog']}.{archetype['target_schema']}.{archetype['target_table']}")
print("=" * 60)

full_table = f"{archetype['target_catalog']}.{archetype['target_schema']}.{archetype['target_table']}"
 
df_verify = (
    spark.table(full_table)
    .filter(col("_source_file") == BAD_RECORDS_PATH)
    .orderBy(col("_ingestion_ts").desc())
)
 
print(f"Registros insertados en esta ejecución: {df_verify.count()}")
display(df_verify.limit(10))



2026-05-27 04:06:21,760 [INFO]   [WRITE BATCH] → fintech_finpay.bronze.users | mode=append | registros: 333
2026-05-27 04:06:24,310 [INFO]   [PROPS] Table properties aplicadas: ['delta.autoOptimize.optimizeWrite', 'delta.autoOptimize.autoCompact']
2026-05-27 04:06:24,311 [INFO] ✅ [WRITE BATCH] 333 registros insertados en fintech_finpay.bronze.users



    REPORTE REPROCESO BAD RECORDS — users
  Bad records leídos   : 333
  Registros insertados : 333
  Tabla destino        : fintech_finpay.bronze.users
🔍 Registros insertados en esta ejecución: 333


user_id,full_name,document_id,email,phone,country,segment,registration_date,_source_file,_ingestion_ts,_source_name,_source_format,_schema_version
USR-002758,Ana María Sáenz,23011033,anamaría.sáenz31@gmail.com,+51913647507,MX,estandar,2020-11-20,/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/,2026-05-27T04:06:22.672Z,users,txt,1
USR-003586,Miriam Elizondo,66749009,miriamelizondo42@yahoo.com,(51)936451132,PE,nuevo,2021-01-19,/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/,2026-05-27T04:06:22.672Z,users,txt,1
USR-004053,Jaqueline Escobar Sanches,91861579,jaqueline.escobar.sa73@gmail.com,(51)959262812,AR,PREMIUM,2020-04-04,/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/,2026-05-27T04:06:22.672Z,users,txt,1
USR-002472,Rosalia Alonso Concepción,2999933,rosaliaalonso.concep45@hotmail.com,+51948524865,MX,premium,2021-04-05,/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/,2026-05-27T04:06:22.672Z,users,txt,1
USR-001828,Dra. Yuridia Munguía,18876507,dra.yuridia.munguía24@hotmail.com,+51965569440,AR,estandar,2021-03-13,/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/,2026-05-27T04:06:22.672Z,users,txt,1
USR-003438,Cristobal Pascual Mondragón Villaseñor,87398842,cristobal.pascual.mo23@outlook.com,956555897,AR,premium,2023-01-28,/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/,2026-05-27T04:06:22.672Z,users,txt,1
USR-003043,Rosa Mares,81501158,rosamares89@hotmail.com,968426834,AR,nuevo,05/09/2022,/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/,2026-05-27T04:06:22.672Z,users,txt,1
USR-001632,Salma Matías,59909962,salma.matías48@hotmail.com,,MX,estandar,24/03/2021,/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/,2026-05-27T04:06:22.672Z,users,txt,1
USR-001210,Dra. Soledad Garica,0244464,dra..soledad.garica18@outlook.com,,PE,estandar,2021-06-03,/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/,2026-05-27T04:06:22.672Z,users,txt,1
USR-000713,Abelardo Mares,06068699,abelardomares38@yahoo.com,+51920662614,CL,premium,2021-04-28,/Volumes/fintech_finpay/default/vol_metadata/bad_records/users/bad_records/,2026-05-27T04:06:22.672Z,users,txt,1


In [0]:
# =============================================================================
# CELDA 10 — MOVER BAD RECORDS A CARPETA DE PROCESADOS
# Solo se ejecuta si records_written > 0 (escritura exitosa).
# Mueve los archivos a processed/ para no reprocesarlos accidentalmente.
# No elimina — mantiene trazabilidad completa.
# =============================================================================

if records_written > 0:

    import os

    BAD_RECORDS_FILES = BAD_RECORDS_PATH        # /Volumes/.../bad_records/users/bad_records/
    PROCESSED_PATH    = BAD_RECORDS_PATH.replace("bad_records/", "re_processed/")

    # Crear carpeta de procesados si no existe
    dbutils.fs.mkdirs(PROCESSED_PATH)

    # Listar archivos en bad_records
    files = dbutils.fs.ls(BAD_RECORDS_FILES)

    moved  = 0
    errors = 0

    for f in files:
        dest = f.path.replace("bad_records/", "re_processed/")
        try:
            dbutils.fs.mv(f.path, dest)
            print(f"  Movido: {f.name} → processed/")
            moved += 1
        except Exception as e:
            print(f"  Error moviendo {f.name}: {e}")
            errors += 1

    print()
    print(f"Archivos movidos a processed/ : {moved}")
    if errors > 0:
        print(f" Errores al mover             : {errors}")

else:
    print("  No se movieron archivos — records_written == 0. Revisa el reproceso.")

  Movido: part-00000-f4464ee8-bf6c-452f-9da8-91b30b662d4b → processed/
  Movido: part-00001-194a7883-96a5-4d11-bf5e-dfb55ae2496d → processed/

Archivos movidos a processed/ : 2
